In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif

In [ ]:
dfTrain = pd.read_csv('train.csv')
dfTest = pd.read_csv("test.csv")
model = LogisticRegression(max_iter=1000, class_weight='balanced')

In [ ]:
dfTrain.head()

In [ ]:
dfTrain = dfTrain.fillna('Unknown')

In [ ]:
dfTrain['HomePlanet'].unique()

In [ ]:
# mapping the values to numbers

dfTrain['HomePlanet'] = dfTrain['HomePlanet'].map({'Europa': 0, 'Earth': 1, 'Mars': 2, 'Unknown': 3})
dfTrain['CryoSleep'] = dfTrain['CryoSleep'].map({False: 0, True: 1, 'Unknown': 2})
dfTrain['Destination'] = dfTrain['Destination'].map({'TRAPPIST-1e': 0, 'PSO J318.5-22': 1, '55 Cancri e': 2, 'Unknown': 3})
dfTrain['VIP'] = dfTrain['VIP'].map({False: 0, True: 1, 'Unknown': 2})
dfTrain['Transported'] = dfTrain['Transported'].map({False: 0, True: 1})


In [ ]:
dfTrain.head()

In [ ]:
features = ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
target = ['Transported']

for col in features:
    dfTrain[col] = dfTrain[col].replace('Unknown', 0).astype(float)

In [ ]:
x = dfTrain[features]
y = dfTrain[target]

In [ ]:
# Calculating the mutual information to choose the right collumns

mutualInformation = mutual_info_classif(x, y, random_state=0)

mutualInformationSeries = pd.Series(mutualInformation, index=x.columns)

mutualInformationSeries = mutualInformationSeries.sort_values(ascending=False)

In [ ]:
print(mutualInformationSeries)

In [ ]:
dfTrain['Total_Spent'] = dfTrain['Spa'] + dfTrain['RoomService'] + dfTrain['VRDeck'] + dfTrain['ShoppingMall'] + dfTrain['FoodCourt']

In [ ]:
dfTrain.head()

In [ ]:
mutualInformationTotalSpent = mutual_info_classif(dfTrain[['Total_Spent']], y, random_state=0)
mutualInformationTotalSpentSeries = pd.Series(mutualInformationTotalSpent)
print(mutualInformationTotalSpentSeries)

In [ ]:
#features = ['CryoSleep', 'Total_Spent', 'HomePlanet']
x = dfTrain[features]
y = dfTrain[target]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

In [ ]:
model.fit(x_train, y_train.values.ravel())

In [ ]:
predictions = model.predict(x_test)

In [ ]:
cm = confusion_matrix(y_test, predictions)

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=["Wasn't transported (0)", 'Was transported (1)'], 
            yticklabels=["Wasn't transported", 'Was transported (1)'])

plt.xlabel('Predicted Outcome')
plt.ylabel('Actual Outcome')
plt.title('Transported predictions results')
plt.show()

In [ ]:
# mapping the values to numbers

dfTest['HomePlanet'] = dfTest['HomePlanet'].map({'Europa': 0, 'Earth': 1, 'Mars': 2, 'Unknown': 3})
dfTest['CryoSleep'] = dfTest['CryoSleep'].map({False: 0, True: 1, 'Unknown': 2})
dfTest['Destination'] = dfTest['Destination'].map({'TRAPPIST-1e': 0, 'PSO J318.5-22': 1, '55 Cancri e': 2, 'Unknown': 3})
dfTest['VIP'] = dfTest['VIP'].map({False: 0, True: 1, 'Unknown': 2})
# dfTest['Transported'] = dfTest['Transported'].map({False: 0, True: 1})


In [ ]:
dfTest['Total_Spent'] = dfTest['Spa'] + dfTest['RoomService'] + dfTest['VRDeck'] + dfTest['ShoppingMall'] + dfTest['FoodCourt']

In [ ]:
xTest = dfTest[features]

In [ ]:
print(features)

In [ ]:
print(xTest.isna().sum())

In [ ]:
dfTest = dfTest.fillna('Unknown')

for col in features:
    dfTest[col] = dfTest[col].replace('Unknown', 0).astype(float)

In [ ]:
xTest = dfTest[features]

In [ ]:
predictionsTest = model.predict(xTest)

In [ ]:
print(predictionsTest)

In [ ]:
submission = pd.DataFrame({
    'PassengerId': dfTest['PassengerId'],
    'Transported': predictionsTest
})

In [ ]:
submission['Transported'] = submission['Transported'].astype(bool)

In [ ]:
submission.to_csv('submission.csv', index=False)